# Graph RAG — Retrieval Pipeline

True Graph RAG: Query → LLM generates Cypher → Neo4j → LLM answers

| Step | What happens |
|------|--------------|
| 1 | Imports & credentials loaded from `.env` |
| 2 | Fetch live schema from Neo4j (labels, rel types, property keys) |
| 3 | LLM generates a Cypher query from the question + schema |
| 4 | Execute Cypher on Neo4j Aura and get raw records |
| 5 | LLM refines raw records into a natural language answer |
| 6 | Run example queries end-to-end |

## Step 1 — Imports & Configuration

In [1]:
import os, re
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

load_dotenv()

NEO4J_URI          = os.environ["NEO4J_URI"]
NEO4J_USERNAME     = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD     = os.environ["NEO4J_PASSWORD"]
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

LLM_MODEL = "openai/gpt-4o-mini"

openai_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

print("✅ Step 1 complete — imports loaded, credentials set")

✅ Step 1 complete — imports loaded, credentials set


## Step 2 — Fetch Live Schema from Neo4j

Instead of hard-coding the schema, we query Neo4j directly so the LLM always
sees the current state of the graph.

In [2]:
def fetch_schema_from_neo4j() -> str:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session() as session:
        labels    = [r["label"]           for r in session.run("CALL db.labels()")]
        rel_types = [r["relationshipType"] for r in session.run("CALL db.relationshipTypes()")]
        prop_keys = [r["propertyKey"]      for r in session.run("CALL db.propertyKeys()")]
    driver.close()

    schema = (
        f"Node labels: {labels}\n"
        f"Relationship types: {rel_types}\n"
        f"Property keys: {prop_keys}"
    )
    print(f"✅ Step 2 complete — live schema fetched")
    print(f"   Labels    : {labels}")
    print(f"   Rel types : {rel_types}")
    return schema

SCHEMA = fetch_schema_from_neo4j()

✅ Step 2 complete — live schema fetched
   Labels    : []
   Rel types : []


## Step 3 — LLM Generates Cypher

The LLM receives the live schema and the user question, then returns a raw
Cypher query. Markdown fences are stripped in case the LLM adds them.

In [3]:
CYPHER_GEN_PROMPT = """
You are a Neo4j Cypher expert. Given the graph schema and a user question,
write a Cypher query to retrieve the relevant information.

Graph Schema:
{schema}

Rules:
- Return ONLY the raw Cypher query, no explanation, no markdown, no code fences
- Use MATCH and OPTIONAL MATCH to traverse relationships
- Always RETURN meaningful node properties, not entire nodes
- If the question is broad, return up to 20 results using LIMIT 20
- Use case-insensitive matching with toLower() when filtering by name

User Question: {question}

Cypher Query:
"""

def generate_cypher(question: str, schema: str) -> str:
    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": CYPHER_GEN_PROMPT.format(schema=schema, question=question)}],
        temperature=0,
    )
    cypher = resp.choices[0].message.content.strip()
    cypher = re.sub(r"^```(?:cypher)?\s*", "", cypher)
    cypher = re.sub(r"\s*```$", "", cypher)
    print(f"✅ Step 3 complete — generated Cypher:")
    print(f"   {cypher}")
    return cypher

print("Step 3 function defined — will run in Step 6")

Step 3 function defined — will run in Step 6


## Step 4 — Execute Cypher on Neo4j

Runs the generated Cypher and returns raw records as a list of dicts.
Failures are caught and return an empty list so the pipeline continues.

In [4]:
def execute_cypher(cypher: str) -> list[dict]:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    try:
        with driver.session() as session:
            records = [dict(r) for r in session.run(cypher)]
        print(f"✅ Step 4 complete — {len(records)} records returned")
        return records
    except Exception as e:
        print(f"❌ Step 4 failed — Cypher execution error: {e}")
        return []
    finally:
        driver.close()

print("Step 4 function defined — will run in Step 6")

Step 4 function defined — will run in Step 6


## Step 5 — LLM Generates Final Answer

The LLM receives the original question and the raw Neo4j records, then
produces a concise natural language answer.

In [5]:
ANSWER_PROMPT = """
You are a helpful assistant. The user asked a question and we retrieved the
following raw data from a knowledge graph to help answer it.

User Question: {question}

Raw Graph Data:
{raw_data}

Using only the data above, provide a clear and concise answer.
If the data is empty or insufficient, say so honestly.
"""

def generate_answer(question: str, records: list[dict]) -> str:
    raw_data = "\n".join(str(r) for r in records) if records else "No results found in the graph."
    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": ANSWER_PROMPT.format(question=question, raw_data=raw_data)}],
    )
    print("✅ Step 5 complete — final answer generated")
    return resp.choices[0].message.content.strip()

print("Step 5 function defined — will run in Step 6")

Step 5 function defined — will run in Step 6


## Step 6 — Run Example Queries

The `graph_rag` orchestrator chains Steps 2–5 for any natural language question.
Edit `queries` below to test your own questions.

In [6]:
def graph_rag(question: str) -> str:
    print(f"\nQuery: \"{question}\"")
    print("-" * 60)
    schema  = fetch_schema_from_neo4j()
    cypher  = generate_cypher(question, schema)
    records = execute_cypher(cypher)
    answer  = generate_answer(question, records)
    print("\n✅ Done")
    return answer

queries = [
    "what are roles we have in our company?",
    "who is the manager of Alice?",
]

for q in queries:
    answer = graph_rag(q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("=" * 60)


Query: "what are roles we have in our company?"
------------------------------------------------------------
✅ Step 2 complete — live schema fetched
   Labels    : []
   Rel types : []
✅ Step 3 complete — generated Cypher:
   MATCH (n) WHERE toLower(n.name) CONTAINS 'role' RETURN n.name, n.data, n.id LIMIT 20
✅ Step 4 complete — 0 records returned
✅ Step 5 complete — final answer generated

✅ Done

Q: what are roles we have in our company?
A: There are no results found in the graph regarding the roles in your company. Therefore, I cannot provide information on that topic.

Query: "who is the manager of Alice?"
------------------------------------------------------------
✅ Step 2 complete — live schema fetched
   Labels    : []
   Rel types : []
✅ Step 3 complete — generated Cypher:
   MATCH (a {name: 'Alice'})-[:MANAGES]->(m) 
RETURN m.name, m.id, m.data, m.style 
LIMIT 20


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MANAGES` does not exist in database `53776d05`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=29, offset=28>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 28, 'line': 1, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a {name: 'Alice'})-[:MANAGES]->(m) \nRETURN m.name, m.id, m.data, m.style \nLIMIT 20"


✅ Step 4 complete — 0 records returned
✅ Step 5 complete — final answer generated

✅ Done

Q: who is the manager of Alice?
A: There is no information available regarding the manager of Alice in the provided data.
